**Question 5:**

Notre ALGO prend en entrée un employé de référence, et regarde chaque machine qui comporte un produit et sur laquelle notre employé de référence peut venir travailler, puis calcule pour chacune d'entre elles le temps d'attente moyen de la pièce qui se trouve dedans avant qu'un autre employé (que celui qui est disponible) puisse s'en occuper.

Pour faire ce calcul, on regarde chaque machine sur laquelle travaille déjà un employé, et on regarde pour chacune d'entre elles, le temps moyen de traitement de la pièce qui se trouve sur cette machine; ce à quoi on soustrait le temps déjà passé dans par cette pièce sur cette machine (on pourra obtenir un score négatif ce qui n'est pas dérangeant, car un tel score indiquerait que l'employé à cette machine est sur le point d'être libéré), ce qui nous donnera donc en moyenne le temps qu'il faudra attendre pour que cet employé se libère. Ensuite, on regarde toutes les machines qui correspondent aux capacités de l'employé de référence et où il y a une pièce en attente, et on évalue pour chacune d'entre elles le temps qu'elle aura à attendre si l'employé de référence ne s'en occupe pas, en calculant le plus petit temps d'attente pour qu'un employé qui a la qualification pour s'occuper de cette pièce se libère.

Ensuite, on envoie l'employé de référence sur la machine avec le temps d'attente le plus élevé.

Ainsi, notre ALGO favorise les produits qui cont attendre longtemps avant d'être pris en charge, ce qui va permettre de réduire le temps d'attente moyen des produits.

**Question 6:**

Tout d'abord, on a besoin de la fonction utilitaire suivante pour calculer le temps moyen de libération d'un employé:

In [ ]:
function temps_restant_employe(atelier::Atelier, emp_id::Int)

    emp = atelier.employes[emp_id]

    # Employé libre
    if !emp.est_occupe
        return 0.0
    end

    m_id = emp.machine_assignee

    # sécurité
    if m_id === nothing
        return 0.0
    end

    machine = atelier.machines[m_id]

    if machine.en_service === nothing
        return 0.0
    end

    p = machine.en_service

    # temps moyen théorique
    a, b = TEMPS[(p.type, m_id)]
    temps_moyen = (a + b) / 2

    # temps déjà passé sur la machine
    temps_deja = now(atelier.sim) - p.arrivee_machine

    return temps_moyen - temps_deja
end

Et ensuite on remplace la fonction choisir_machine dans le simulateur donné par:

In [ ]:
function choisir_machine(atelier::Atelier, emp_id::Int,
                         mode_algo::String,
                         mode_choix::String)

    emp_ref = atelier.employes[emp_id]

    meilleure_machine = nothing
    pire_attente = -Inf

    # On regarde toutes les machines compatibles
    for m_id in emp_ref.qualifications

        machine = atelier.machines[m_id]

        # il faut une pièce en attente
        if isempty(machine.file_attente)
            continue
        end

        # --- estimation du temps d'attente si l'employé courant NE prend PAS cette machine

        meilleur_temps = Inf

        # chercher le premier autre employé capable
        for autre_id in 1:length(atelier.employes)

            # ne pas prendre l'employé actuel
            if autre_id == emp_id
                continue
            end

            autre = atelier.employes[autre_id]

            # qualification nécessaire ?
            if !(m_id in autre.qualifications)
                continue
            end

            # temps avant libération
            t = temps_restant_employe(atelier, autre_id)

            if t < meilleur_temps
                meilleur_temps = t
            end
        end

        # aucun autre employé disponible
        if meilleur_temps == Inf
            meilleur_temps = 1e9
        end

        log_event(atelier,
            "Machine $m_id : attente estimée = $(round(meilleur_temps,digits=3))")

        # on choisit la machine
        # dont l'attente future serait la pire
        if meilleur_temps > pire_attente
            pire_attente = meilleur_temps
            meilleure_machine = m_id
        end
    end

    if meilleure_machine !== nothing
        log_event(atelier,
            "Employé $emp_id choisit machine $meilleure_machine " *
            "(attente évitée = $(round(pire_attente,digits=3)))")
    end

    return meilleure_machine
end

Résultats: 

**Question 7:**

**Question 8:**

**Question 9:**

**Question 10:**